# Embedding Validation
This notebook tests the `FeatureProcessor` from `src/embedder.py` on the new Universal Training Table.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append('../')
from src.embedder import FeatureProcessor

processor = FeatureProcessor()

## 1. Load Universal Training Table

In [2]:
# Load data
data_path = "../data/processed/UNIVERSAL_training.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} rows")
df.head()

## 2. Test Embedding Generation
Using `all-MiniLM-L6-v2` local model.

In [3]:
df_emb = processor.embed_summaries(df.head(10)) # Small sample for validation
df_emb.head()

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between first two companies to verify embedding semantic quality
sim = cosine_similarity([df_emb.iloc[0].filter(like='nlp_')], [df_emb.iloc[1].filter(like='nlp_')])
print(f"Similarity between {df_emb.iloc[0]['ticker']} and {df_emb.iloc[1]['ticker']}: {sim[0][0]:.4f}")

## 3. Test Feature Preservation & Metadata Dropping
Ensuring that metadata is removed but features like `sector` are preserved for the Valuation Engine.

In [5]:
# Demonstrate dropping metadata while keeping 'sector'
df_ml = processor.drop_extra_columns(df_emb)

print(f"Columns after dropping metadata: {df_ml.columns.tolist()[:10]}...")
print(f"Sector preserved: {'sector' in df_ml.columns}")
print(f"Remaining columns: {len(df_ml.columns)}")

df_ml.head()

## 4. Verify Final Universal Embedded Dataset
Verify the output of the full pipeline.

In [7]:
universal_embedded = "../data/processed/UNIVERSAL_embedded.parquet"

if os.path.exists(universal_embedded):
    df_final = pd.read_parquet(universal_embedded)
    print(f"Universal Embedded Table Shape: {df_final.shape}")
    print(f"Critical features present: {'sector' in df_final.columns and 'nlp_0' in df_final.columns}")
    display(df_final.head(2))
else:
    print("Universal embedded file not found. Run 'poetry run python src/embedder.py' first.")